# Treino e avaliação da MobileNetV2Notebook para executar localmente no Jupyter ou VS Code. O treino usa o dataset TrashNet já extraído no projeto. Um runtime com GPU acelera o processo, mas não é obrigatório.

In [ ]:
from pathlib import Pathraizes_candidatas = [Path.cwd(), *Path.cwd().parents]PROJECT_DIR = next(    (raiz for raiz in raizes_candidatas if (raiz / "scripts/train_classifier.py").is_file()),    None,)if PROJECT_DIR is None:    raise FileNotFoundError("Abra o notebook a partir do projeto iot-ai.")DATA_DIR = PROJECT_DIR / "trashnet/dataset-resized/dataset-resized"OUTPUT_DIR = PROJECT_DIR / "models"if not DATA_DIR.is_dir():    raise FileNotFoundError(f"Dataset não encontrado: {DATA_DIR.resolve()}")print("Projeto:", PROJECT_DIR)print("Dataset:", DATA_DIR)print("Saída dos modelos:", OUTPUT_DIR)

## Ambiente PythonUse um kernel com as dependências de `requirements-train.txt` instaladas. Python 3.11 ou 3.12 é recomendado; a MobileNetV2 usa GPU quando disponível e continua executável em CPU.

In [ ]:
import sysimport torchprint("Python do kernel:", sys.executable)print("PyTorch:", torch.__version__)print("Dispositivo:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Conferir as classesO dataset local deve conter as seis subpastas. As imagens não precisam ser copiadas para o Drive nem baixadas novamente.

In [ ]:
classes_esperadas = {"cardboard", "glass", "metal", "paper", "plastic", "trash"}classes_encontradas = {pasta.name for pasta in DATA_DIR.iterdir() if pasta.is_dir()}if not classes_esperadas.issubset(classes_encontradas):    raise ValueError(f"Classes ausentes: {sorted(classes_esperadas - classes_encontradas)}")print("Classes encontradas:", sorted(classes_esperadas))print("Pronto para treinar.")

## Treinar, avaliar e exportarA divisão é estratificada em treino, validação e teste. O relatório no teste TrashNet fica separado da avaliação opcional em fotos reais. Para avaliar fotos próprias, organize-as em `data/raw/real/<classe>/` e mantenha-as fora do treino.

In [ ]:
import sys

sys.path.insert(0, str(PROJECT_DIR))
argumentos = [
    "train_classifier.py",
    "--data", str(DATA_DIR),
    "--output", str(OUTPUT_DIR),
    "--epochs", "10",
]
REAL_DATA_DIR = PROJECT_DIR / "data/raw/real"
if REAL_DATA_DIR.is_dir():
    argumentos += ["--real-data", str(REAL_DATA_DIR)]

AVALIAR_CHECKPOINT_EXISTENTE = False
if AVALIAR_CHECKPOINT_EXISTENTE:
    argumentos += ["--evaluate-only"]

sys.path.insert(0, str(PROJECT_DIR))
from scripts.train_classifier import main

argumentos_originais = sys.argv
try:
    sys.argv = argumentos
    main()
finally:
    sys.argv = argumentos_originais

### Se o treino parou depois de salvar o checkpoint

Defina `AVALIAR_CHECKPOINT_EXISTENTE = True` na célula de treino e execute-a novamente. O script reutiliza `models/mobilenetv2_best.pth` para calcular as métricas de teste e exportar o ONNX, sem repetir as épocas.

In [ ]:
import json

relatorio = json.loads((OUTPUT_DIR / "evaluation.json").read_text(encoding="utf-8"))
print("Tamanhos das divisões:", relatorio["split_sizes"])
print("Acurácia no teste TrashNet:", round(relatorio["test"]["accuracy"], 4))
print("Macro F1 no teste TrashNet:", round(relatorio["test"]["classification_report"]["macro avg"]["f1-score"], 4))
print("Avaliação em fotos reais:", relatorio["real_photos"]["status"])
print("Arquivos exportados:", sorted(p.name for p in OUTPUT_DIR.iterdir()))

## Interpretação

Registre os valores produzidos pelo treino, sem estimá-los antes. O TrashNet tem imagens em cenários controlados; métricas nesse conjunto não demonstram, por si só, o desempenho em fotos reais. Os artefatos `mobilenetv2.onnx` e `metadata.json` são necessários para executar a API FastAPI localmente.